In [22]:
import pandas as pd

df = pd.read_csv("data/router_data.csv")
df[:10]

,task_id,query,query_embedding,ground_truth,metric,llm,effect,cost,task_description,task_description_embedding
0,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,LLaMA-3 (8b),0.191304,0.000474,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
1,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,Mixtral-8x7B,0.289157,0.492417,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
2,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,NousResearch,0.256757,0.861374,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
3,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,NaN,0.196078,0.123460,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
4,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,Mistral-7b,0.245161,0.000474,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
5,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,LLaMA-3 (70b),0.184100,0.861374,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
6,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,LLaMA-3-Turbo (8b),0.200913,0.000474,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
7,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,LLaMA-3-Turbo (70b),0.189655,0.861374,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
8,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,Llama-3.1-Turbo (70b),0.162162,0.861374,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...
9,alpaca_data,Give three tips for staying healthy.,[[-7.58798886e-03 9.38545167e-03 1.12636592e...,1.Eat a balanced diet and make sure to include...,f1_score,Qwen-1.5 (72b),0.241546,0.861374,The Alpaca dataset is designed for instruction...,[[ 1.09750889e-02 -5.29433116e-02 -3.60085852e...


In [18]:
import pandas as pd
from scipy.special import softmax

df = pd.read_csv("data/router_data_nlgFA.csv")

idx = 3
block = df[idx*6 : idx*6+6]

# Compute values
max_effect = block[['llm', 'effect']].loc[block['effect'].idxmax()]
max_cost   = block[['llm', 'cost']].loc[block['cost'].idxmax()]
min_cost   = block[['llm', 'cost']].loc[block['cost'].idxmin()]

# Build a results table
results = pd.DataFrame([
    {'metric': 'max_effect', 'llm': max_effect['llm'], 'value': max_effect['effect']},
    {'metric': 'min_cost',   'llm': min_cost['llm'],   'value': min_cost['cost']},
])
results

,metric,llm,value
0,max_effect,Qwen-2.5 (7b),0.104712
1,min_cost,Gemma-3N (e4b),0.002091


In [21]:
idx = 7
block = df[idx*6 : idx*6+6].reset_index(drop=True)

# ---------------------------
# Normalize effect & cost
# ---------------------------
effect = block["effect"]
cost   = block["cost"]

effect_n = (effect - effect.min()) / (effect.max() - effect.min() + 1e-9)
cost_n   = (cost   - cost.min())   / (cost.max()   - cost.min()   + 1e-9)

# ---------------------------
# Compute scores for 3 scenarios
# ---------------------------

# 1️⃣ Performance First (effect only)
score_perf = effect_n

# 2️⃣ Balance (your formula)
score_bal  = 0.6 * effect_n - 0.4 * cost_n

# 3️⃣ Cost First (cost only, reversed → minimizing cost)
score_cost = -cost_n

# Temperature softmax (your formula)
T = 0.15
soft_perf = softmax(score_perf / T)
soft_bal  = softmax(score_bal  / T)
soft_cost = softmax(score_cost / T)

block["score_perf"] = soft_perf
block["score_bal"]  = soft_bal
block["score_cost"] = soft_cost

# ---------------------------
# Extract winners (ground truth)
# ---------------------------
winner_perf = block.loc[block["score_perf"].idxmax(), ["llm", "score_perf"]]
winner_bal  = block.loc[block["score_bal"].idxmax(),  ["llm", "score_bal"]]
winner_cost = block.loc[block["score_cost"].idxmax(), ["llm", "score_cost"]]

# ---------------------------
# Build final results table
# ---------------------------
results = pd.DataFrame([
    {"scenario": "Performance First", "llm": winner_perf["llm"], "score": winner_perf["score_perf"]},
    {"scenario": "Balance",           "llm": winner_bal["llm"],  "score": winner_bal["score_bal"]},
    {"scenario": "Cost First",        "llm": winner_cost["llm"], "score": winner_cost["score_cost"]},
])

results


,scenario,llm,score
0,Performance First,Gemma-3N (e4b),0.564235
1,Balance,Gemma-3N (e4b),0.647337
2,Cost First,Gemma-3N (e4b),0.559193


In [1]:
import pandas as pd

df = pd.read_csv("data/MIZAN_PersianNLG/feedback/router_data.csv")
idx = 3
df[idx*6 : idx*6+6]

,task_id,query,query_embedding,ground_truth,metric,llm,effect,cost,task_description,task_description_embedding,avg_feedback
18,question-generation_PersianQA,بدگُمانی یا پارانویا در تعریف عام آن، غریزه ی...,[ 1.18435510e-02 3.77712138e-02 7.73427403e-...,غریزه یا فرایندی از فکر است که تحت تأثیر اضطرا...,f1_score,Mistral-7b,0.102941,0.264968,The question generation task assesses the abil...,[-6.98172348e-03 2.19651796e-02 -2.53958404e-...,5.000000
19,question-generation_PersianQA,بدگُمانی یا پارانویا در تعریف عام آن، غریزه ی...,[ 1.18435510e-02 3.77712138e-02 7.73427403e-...,غریزه یا فرایندی از فکر است که تحت تأثیر اضطرا...,f1_score,LLlama-3.1 (8B),0.080292,0.234734,The question generation task assesses the abil...,[-6.98172348e-03 2.19651796e-02 -2.53958404e-...,3.333333
20,question-generation_PersianQA,بدگُمانی یا پارانویا در تعریف عام آن، غریزه ی...,[ 1.18435510e-02 3.77712138e-02 7.73427403e-...,غریزه یا فرایندی از فکر است که تحت تأثیر اضطرا...,f1_score,Mixtral-8x7B,0.000000,0.869652,The question generation task assesses the abil...,[-6.98172348e-03 2.19651796e-02 -2.53958404e-...,4.333333
21,question-generation_PersianQA,بدگُمانی یا پارانویا در تعریف عام آن، غریزه ی...,[ 1.18435510e-02 3.77712138e-02 7.73427403e-...,غریزه یا فرایندی از فکر است که تحت تأثیر اضطرا...,f1_score,gpt-oss-20B,0.098901,0.107442,The question generation task assesses the abil...,[-6.98172348e-03 2.19651796e-02 -2.53958404e-...,4.333333
22,question-generation_PersianQA,بدگُمانی یا پارانویا در تعریف عام آن، غریزه ی...,[ 1.18435510e-02 3.77712138e-02 7.73427403e-...,غریزه یا فرایندی از فکر است که تحت تأثیر اضطرا...,f1_score,Qwen-2.5 (7b),0.104712,0.416139,The question generation task assesses the abil...,[-6.98172348e-03 2.19651796e-02 -2.53958404e-...,3.666667
23,question-generation_PersianQA,بدگُمانی یا پارانویا در تعریف عام آن، غریزه ی...,[ 1.18435510e-02 3.77712138e-02 7.73427403e-...,غریزه یا فرایندی از فکر است که تحت تأثیر اضطرا...,f1_score,Gemma-3N (e4b),0.075862,0.002091,The question generation task assesses the abil...,[-6.98172348e-03 2.19651796e-02 -2.53958404e-...,1.333333


In [ ]:


# # ============================================
# # Normalize effect & cost
# # ============================================
# effect = block["effect"].astype(float)
# cost   = block["cost"].astype(float)

# effect_n = (effect - effect.min()) / (effect.max() - effect.min() + 1e-12)
# cost_n   = (cost   - cost.min())   / (cost.max()   - cost.min()   + 1e-12)




# # ============================================
# # Feedback weighting
# # ============================================
# alpha = 0.70  # same α you use when training the router


# # ============================================
# # Compute baseline scenario scores
# # ============================================

# # 1️⃣ Performance First
# score_perf = effect_n.copy()

# # 2️⃣ Balance
# score_bal = (0.6 * effect_n) - (0.4 * cost_n)

# # 3️⃣ Cost First
# score_cost = -cost_n

# # ============================================
# # Inject feedback (same rule as router code)
# # ============================================
# score_perf = (1 - alpha) * score_perf + alpha * feedback_n
# score_bal  = (1 - alpha) * score_bal  + alpha * feedback_n
# score_cost = (1 - alpha) * score_cost + alpha * feedback_n


# # ============================================
# # Apply softmax temperature scaling
# # ============================================
# T = 0.15
# soft_perf = softmax(score_perf / T)
# soft_bal  = softmax(score_bal  / T)
# soft_cost = softmax(score_cost / T)

# block["score_perf"] = soft_perf
# block["score_bal"]  = soft_bal
# block["score_cost"] = soft_cost


# # ============================================
# # Extract winners
# # ============================================
# winner_perf = block.loc[block["score_perf"].idxmax(), ["llm", "score_perf"]]
# winner_bal  = block.loc[block["score_bal"].idxmax(),  ["llm", "score_bal"]]
# winner_cost = block.loc[block["score_cost"].idxmax(), ["llm", "score_cost"]]


# # ============================================
# # Build final results table
# # ============================================
# results = pd.DataFrame([
#     {"scenario": "Performance First", "llm": winner_perf["llm"], "score": winner_perf["score_perf"]},
#     {"scenario": "Balance",           "llm": winner_bal["llm"],  "score": winner_bal["score_bal"]},
#     {"scenario": "Cost First",        "llm": winner_cost["llm"], "score": winner_cost["score_cost"]},
# ])

def compute_winners_table(block, alpha=0.4, T=0.15):
    """
    Compute winning LLMs under different scenarios with feedback and softmax scaling.

    Parameters:
        block: pd.DataFrame (must contain ['effect', 'cost', 'feedback_n', 'llm'])
        alpha: float (weight of feedback in score combination)
        T: float (softmax temperature)

    Returns:
        results: pd.DataFrame (scenario, llm, score)
    """
    # Normalizing effect and cost
    effect_n = (block["effect"] - block["effect"].min()) / (block["effect"].max() - block["effect"].min() + 1e-12)
    cost_n = (block["cost"] - block["cost"].min()) / (block["cost"].max() - block["cost"].min() + 1e-12)
    
    # ============================================
    # Load and normalize feedback (same as router)
    # ============================================
    
    if "avg_feedback" in block.columns:
        feedback_list = block["avg_feedback"].fillna(0).astype(float).values

        # Normalize feedback if > 1.0 (assumes 1–5 rating)
        if feedback_list.max() > 1.0:
            FEEDBACK_MIN_SCORE = 1.0
            FEEDBACK_NORMALIZED_RANGE = 4.0  # (5 - 1)
            feedback_list = (feedback_list - FEEDBACK_MIN_SCORE) / FEEDBACK_NORMALIZED_RANGE

        # Feedback min-max normalization
        feedback_n = (feedback_list - feedback_list.min()) / (feedback_list.max() - feedback_list.min() + 1e-12)

    else:
        feedback_n = np.zeros(len(block))



    # Baseline scores
    score_perf = effect_n.copy()
    score_bal = (0.6 * effect_n) - (0.4 * cost_n)
    score_cost = -cost_n

    # Inject feedback
    score_perf = (1 - alpha) * score_perf + alpha * feedback_n
    score_bal  = (1 - alpha) * score_bal  + alpha * feedback_n
    score_cost = (1 - alpha) * score_cost + alpha * feedback_n

    # Softmax
    def softmax(x):
        x = np.array(x)
        x = x - np.max(x)
        e_x = np.exp(x / T)
        return e_x / (e_x.sum() + 1e-12)

    soft_perf = softmax(score_perf)
    soft_bal = softmax(score_bal)
    soft_cost = softmax(score_cost)

    block = block.copy()
    block["score_perf"] = soft_perf
    block["score_bal"] = soft_bal
    block["score_cost"] = soft_cost

    # Winners
    winner_perf = block.loc[block["score_perf"].idxmax(), ["llm", "score_perf"]]
    winner_bal  = block.loc[block["score_bal"].idxmax(),  ["llm", "score_bal"]]
    winner_cost = block.loc[block["score_cost"].idxmax(), ["llm", "score_cost"]]

    results = pd.DataFrame([
        {"scenario": "Performance First", "llm": winner_perf["llm"], "score": winner_perf["score_perf"]},
        {"scenario": "Balance",           "llm": winner_bal["llm"],  "score": winner_bal["score_bal"]},
        {"scenario": "Cost First",        "llm": winner_cost["llm"], "score": winner_cost["score_cost"]},
    ])
    return results

import pandas as pd
import numpy as np
from scipy.special import softmax

df = pd.read_csv("data/MIZAN_PersianNLG/feedback/router_data.csv")

idx = 7
block = df[idx*6 : idx*6+6].reset_index(drop=True)

alpha = 0.4  # same α you use when training the router
T = 0.15

# Example usage (after block is defined):
results = compute_winners_table(block, alpha=alpha, T=T)

results

,scenario,llm,score
0,Performance First,Gemma-3N (e4b),0.924859
1,Balance,Gemma-3N (e4b),0.931792
2,Cost First,Gemma-3N (e4b),0.910088
